# 🛣️ PathSense: YOLOv8 Cloud Training on Google Colab (NVIDIA T4)
### **7th Semester Minor Project — Department of CS-AI, SKIT Jaipur**

**Team Members:**
1. **Sharafat Khan** (`23ESKCA098`)
2. **Soham Manocha** (`23ESKCA102`)
3. **Sourabh Nagar** (`23ESKCA105`)
4. **Vedic Baurasi** (`23ESKCA119`)

---
### Overview
This notebook fine-tunes **YOLOv8** on the **RDD2022 Indian Road Damage Dataset** (Potholes, Longitudinal Cracks, Transverse Cracks, Alligator Cracks). Once trained, the exported `best.pt` weights will be downloaded to your local PathSense project for real-time edge inference.

## Step 1: Verify GPU Acceleration (NVIDIA T4 / V100)
> *Make sure Runtime is set to GPU: Click **Runtime** > **Change runtime type** > Select **T4 GPU**.*

In [ ]:
!nvidia-smi

## Step 2: Install Ultralytics & Dependencies

In [ ]:
!pip install -q ultralytics roboflow gdown

## Step 3: Connect Google Drive (To access your Dataset)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import zipfile

print("Google Drive successfully mounted!")

## Step 4: Prepare RDD2022 Indian Road Dataset
If your dataset is in Google Drive or you want to download the India dataset subset directly, run this cell:

In [ ]:
# Option A: Copy from Google Drive if you saved a zip in Drive
# e.g. !cp /content/drive/MyDrive/RDD2022_India.zip /content/

# Create dataset directory
!mkdir -p /content/pathsense_dataset/images/train /content/pathsense_dataset/images/val
!mkdir -p /content/pathsense_dataset/labels/train /content/pathsense_dataset/labels/val

# Download benchmark Indian road pothole & damage dataset
!pip install -q kaggle
print("Dataset directories initialized at /content/pathsense_dataset/")

## Step 5: Create `data.yaml` Configuration for RDD2022 India Classes

In [ ]:
yaml_content = """
path: /content/pathsense_dataset
train: images/train
val: images/val

# 4 Road Distress Classes (Official RDD2022 India Benchmark)
names:
  0: longitudinal_crack  # D00
  1: transverse_crack    # D10
  2: alligator_crack     # D20
  3: pothole             # D40
"""

with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("Saved /content/data.yaml successfully!")

## Step 6: Train YOLOv8 on Cloud GPU
We use **YOLOv8 Medium (`yolov8m.pt`)** with domain-specific augmentations (perspective distortion, shadow variations, and motion blur) tailored for Indian roads.

In [ ]:
from ultralytics import YOLO

# 1. Load pretrained YOLOv8 model
model = YOLO('yolov8m.pt')

# 2. Start Cloud Training
results = model.train(
    data='/content/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='pathsense_rdd',
    name='yolov8_india_road',
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    save=True,
    plots=True
)

print("Training Complete!")

## Step 7: Evaluate Validation Metrics (mAP@50 & Confusion Matrix)

In [ ]:
# Validate the model
metrics = model.val()
print(f"Validation mAP@50: {metrics.box.map50:.4f}")
print(f"Validation mAP@50-95: {metrics.box.map:.4f}")

## Step 8: Download `best.pt` to Local Laptop
Run this cell to automatically download your fine-tuned weights file (`best.pt`) directly to your PC, so you can place it in `pathsense/core/`!

In [ ]:
from google.colab import files
import os

best_weights = '/content/pathsense_rdd/yolov8_india_road/weights/best.pt'
if os.path.exists(best_weights):
    print("Downloading fine-tuned weights best.pt...")
    files.download(best_weights)
else:
    print("Trained weights not found at default path, checking runs folder:")
    !find /content -name "best.pt"